In [17]:
import sys
from datasets import load_dataset

sys.path.append('../../')
from src.utils.memory_utils import process_in_chunks

# LOAD DATASETS
user_data = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023", 
    "raw_review_CDs_and_Vinyl", 
    split="full", 
    trust_remote_code=True
)

meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023", 
    "raw_meta_CDs_and_Vinyl", 
    split="full", 
    trust_remote_code=True
)

# FLATTEN LIST COLUMNS IN METADATA
def flatten_lst(col):
    """Flatten lists by joining their elements with a separator (e.g., ', ')"""
    col['features'] = ', '.join(col['features']) if isinstance(col['features'], list) else col['features']
    col['description'] = ', '.join(col['description']) if isinstance(col['description'], list) else col['description']
    col['categories'] = ', '.join(col['categories']) if isinstance(col['categories'], list) else col['categories']
    return col

flattened_meta = meta.map(flatten_lst)

# CONVERT TO PANDAS WITH CHUNK PROCESSING
user_df = process_in_chunks(user_data.to_pandas(), chunk_size=10000)
meta_df = process_in_chunks(flattened_meta.to_pandas(), chunk_size=10000)

# user_df = user_data.to_pandas().fillna(0)
# meta_df = flattened_meta.to_pandas().fillna(0)


Processed 0 rows. Current memory usage: 1544.41 MB
Processed 10000 rows. Current memory usage: 1557.83 MB
Processed 20000 rows. Current memory usage: 1564.72 MB
Processed 30000 rows. Current memory usage: 1560.86 MB
Processed 40000 rows. Current memory usage: 1539.06 MB
Processed 50000 rows. Current memory usage: 1540.73 MB
Processed 60000 rows. Current memory usage: 1542.55 MB
Processed 70000 rows. Current memory usage: 1549.16 MB
Processed 80000 rows. Current memory usage: 1554.70 MB
Processed 90000 rows. Current memory usage: 1561.03 MB
Processed 100000 rows. Current memory usage: 1565.00 MB
Processed 110000 rows. Current memory usage: 1545.41 MB
Processed 120000 rows. Current memory usage: 1552.73 MB
Processed 130000 rows. Current memory usage: 1559.25 MB
Processed 140000 rows. Current memory usage: 1566.05 MB
Processed 150000 rows. Current memory usage: 1571.72 MB
Processed 160000 rows. Current memory usage: 1578.53 MB
Processed 170000 rows. Current memory usage: 1584.38 MB
Proces

In [18]:
user_df['verified_purchase'] = user_df['verified_purchase'].astype(int)
user_df.head(5)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Five Stars,LOVE IT!,[],B002MW50JA,B002MW50JA,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650777000,0,1
1,5.0,Five Stars,LOVE!!,[],B008XNPN0S,B008XNPN0S,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650764000,0,1
2,3.0,Three Stars,Sad there is not the versions with the real/or...,[],B00IKM5N02,B00IKM5N02,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452649885000,0,1
3,3.0,Disappointed,I have listen to The Broadway 1958 Flower Drum...,[],B00006JKCM,B00006JKCM,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1164036864000,3,1
4,5.0,Wonderful melding,Simply great album. One of the best. Marvelous...,[],B00013YRQY,B00013YRQY,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1582090199946,0,0


In [19]:
user_df['rating'] = user_df['rating'].astype('float32')
user_df.groupby(['rating','verified_purchase']).size().reset_index().pivot(columns='rating', index='verified_purchase', values=0)

rating,1.0,2.0,3.0,4.0,5.0
verified_purchase,,,,,
0,81604,63704,117798,288213,996369
1,106123,76853,165893,376301,2554415


In [20]:
user_df2 = user_df[['rating', 'asin', 'parent_asin', 'user_id', 'timestamp', 'text']]
user_df2

,rating,asin,parent_asin,user_id,timestamp,text
0,5.0,B002MW50JA,B002MW50JA,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650777000,LOVE IT!
1,5.0,B008XNPN0S,B008XNPN0S,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650764000,LOVE!!
2,3.0,B00IKM5N02,B00IKM5N02,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452649885000,Sad there is not the versions with the real/or...
3,3.0,B00006JKCM,B00006JKCM,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1164036864000,I have listen to The Broadway 1958 Flower Drum...
4,5.0,B00013YRQY,B00013YRQY,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1582090199946,Simply great album. One of the best. Marvelous...
...,...,...,...,...,...,...
4827268,5.0,B000002VPH,B000002VPH,AHM36UEBOF2I6VH7CGAGHCDDUITQ,1308413175000,I love this cd and love the movie thank u I ha...
4827269,5.0,B000084T18,B000084T18,AHM36UEBOF2I6VH7CGAGHCDDUITQ,1308412875000,I love the cd it play real well and was delive...
4827270,5.0,B004OFWLO0,B004OFWLO0,AHRJPHOI5JHYEQVSDMNX6736QH3Q,1505425559120,Such a great remaster you can fully appreciate...
4827271,1.0,B000GIXIAK,B000GIXIAK,AH4PJ73QN75AJM5VSCT53AOADCGA,1157470110000,What this CD is lacking is a heart. The music...


In [21]:
# drop cols dont provide contextual meaning
user_df = user_df.drop(['images'], axis='columns')

In [22]:
meta_df.head(5)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Digital Music,Release Some Tension,4.601562,112,,Swv ~ Release Some Tension,12.05,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SWV Format: Audio CD,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro...",B000002X4C,None,None,None
1,Digital Music,Rio Angie,5.000000,1,,"Shrimp City Slim (aka Gary Erwin, b. 1953, Chi...",14.98,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Shrimp City Slim (Artist) Format: Audio CD,"CDs & Vinyl, Jazz, Avant Garde & Free Jazz","{""Product Dimensions"": ""5.6 x 0.4 x 4.9 inches...",B00902T10Y,None,None,None
2,Digital Music,Lost in Love,5.000000,9,,,24.99,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Nastyboy Klick Format: Audio CD,"CDs & Vinyl, Rap & Hip-Hop, Gangsta & Hardcore","{""Package Dimensions"": ""4.7 x 4.6 x 0.1 inches...",B00000DALY,None,None,None
3,Digital Music,Somewhere in Time,4.800781,1186,,The 1980 soundtrack to the now classic motion ...,11.55,"{'hi_res': [None, None], 'large': ['https://m....","{'title': [], 'url': [], 'user_id': []}","John Barry (Composer), Barry, John (Comp...","CDs & Vinyl, Soundtracks, Movie Scores","{""Is Discontinued By Manufacturer"": ""No"", ""Lan...",B0000086D1,None,None,None
4,Digital Music,Kimmon Waldruff,5.000000,1,,Solo acoustic fingerstyle guitar.,14.07,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Kimmon Waldruff (Artist) Format: Audio CD,"CDs & Vinyl, Folk","{""Is Discontinued By Manufacturer"": ""No"", ""Pro...",B000S6W7KC,None,None,None


In [23]:
meta_df['main_category'].unique()
meta_df.shape

(701959, 16)

In [24]:
meta_df = meta_df.drop_duplicates(subset=['parent_asin'], keep='last')
meta_df.shape

(701959, 16)

In [25]:
# drop cols dont provide contextual meaning
meta = meta_df.drop(['images', 'features', 'videos', 'store', 'price', 'details', 'bought_together', 'subtitle', 'author'], axis='columns')
meta

,main_category,title,average_rating,rating_number,description,categories,parent_asin
0,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",B000002X4C
1,Digital Music,Rio Angie,5.000000,1,"Shrimp City Slim (aka Gary Erwin, b. 1953, Chi...","CDs & Vinyl, Jazz, Avant Garde & Free Jazz",B00902T10Y
2,Digital Music,Lost in Love,5.000000,9,,"CDs & Vinyl, Rap & Hip-Hop, Gangsta & Hardcore",B00000DALY
3,Digital Music,Somewhere in Time,4.800781,1186,The 1980 soundtrack to the now classic motion ...,"CDs & Vinyl, Soundtracks, Movie Scores",B0000086D1
4,Digital Music,Kimmon Waldruff,5.000000,1,Solo acoustic fingerstyle guitar.,"CDs & Vinyl, Folk",B000S6W7KC
...,...,...,...,...,...,...,...
701954,Digital Music,Forever Gold: British Invasion,3.000000,1,British Invasion ~ Forever Gold: British Invasion,"CDs & Vinyl, International Music, Europe, Brit...",B00005AVHN
701955,Digital Music,"Joan Hammond, Historical Recordings from 1941-49",5.000000,1,,"CDs & Vinyl, Classical",B000T001IM
701956,Digital Music,Come Alive,4.500000,4,The Second Full Length Album from the Winner o...,"CDs & Vinyl, International Music, Africa, Sout...",B00069I6RO
701957,Digital Music,Long Day in the Milky Way,4.398438,16,2020 release from the folk/Americana singer/so...,"CDs & Vinyl, Folk",B08BF2PH1X


In [32]:
# merge df vs meta
merge = user_df2.merge(meta, on='parent_asin', how='right', suffixes=('', '_meta'))
merge.head(5)

,rating,asin,parent_asin,user_id,timestamp,text,main_category,title,average_rating,rating_number,description,categories
0,1.0,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,1.420266e+12,Definitely the weakest of the three albums SWV...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
1,1.0,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,1.379363e+12,Unlike It's About Time and New Beginning this ...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
2,4.0,B000002X4C,B000002X4C,AHAIZWWINONHYU7A3FVCEH75R4CA,9.534267e+11,"Their third and final album as a group, &quot;...",Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
3,5.0,B000002X4C,B000002X4C,AE73R3KLFKXFNWUBOA4JCLT5UWKA,1.427480e+12,thank you,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
4,2.0,B000002X4C,B000002X4C,AFICHVZ62IFAWIBZK3U6B7NZWAVQ,1.289638e+12,When I was trying to decide whether or not to ...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"


In [33]:
merge.columns = [
    'rating', 'iid', 'parent_iid', 'uid', 'timestamp', 'reviews',
    'main_category', 'title', 'average_rating', 'rating_number',
    'description', 'categories'
]

In [34]:
merge.shape

(4827559, 12)

In [35]:
# Check unique counts
merge.iid.nunique(), merge.parent_iid.unique(), merge.uid.nunique()

(701706,
 array(['B000002X4C', 'B00902T10Y', 'B00000DALY', ..., 'B00069I6RO',
        'B08BF2PH1X', 'B000001LW6'], dtype=object),
 1754118)

In [36]:
merge.head(3)

,rating,iid,parent_iid,uid,timestamp,reviews,main_category,title,average_rating,rating_number,description,categories
0,1.0,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,1.420266e+12,Definitely the weakest of the three albums SWV...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
1,1.0,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,1.379363e+12,Unlike It's About Time and New Beginning this ...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
2,4.0,B000002X4C,B000002X4C,AHAIZWWINONHYU7A3FVCEH75R4CA,9.534267e+11,"Their third and final album as a group, &quot;...",Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"


In [38]:
merge_df = merge.copy()
merge_df = merge_df.rename(columns={'title': 'title_review'})
# merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], unit='ms')
merge_df.head(5)

,rating,iid,parent_iid,uid,timestamp,reviews,main_category,title_review,average_rating,rating_number,description,categories
0,1.0,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,1.420266e+12,Definitely the weakest of the three albums SWV...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
1,1.0,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,1.379363e+12,Unlike It's About Time and New Beginning this ...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
2,4.0,B000002X4C,B000002X4C,AHAIZWWINONHYU7A3FVCEH75R4CA,9.534267e+11,"Their third and final album as a group, &quot;...",Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
3,5.0,B000002X4C,B000002X4C,AE73R3KLFKXFNWUBOA4JCLT5UWKA,1.427480e+12,thank you,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
4,2.0,B000002X4C,B000002X4C,AFICHVZ62IFAWIBZK3U6B7NZWAVQ,1.289638e+12,When I was trying to decide whether or not to ...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"


In [ ]:
# filter merge_df with rating >= 4
# merged_df['like'] = merged_df['rating'] >= 4 

# data.drop_duplicates(subset=['asin'])

In [39]:
item_info = merge_df.groupby('iid').agg({"rating_number": 'first'})
user_info = merge_df.groupby('uid').agg({"rating": 'count'})

active_item = item_info[item_info['rating_number'] > 5].index
active_user = user_info[user_info['rating'] > 5].index
active_item.shape, active_user.shape

((452791,), (142425,))

In [40]:
merge_df = merge_df[merge_df['uid'].isin(active_user) & merge_df['iid'].isin(active_item)]

In [41]:
merge_df['label'] = 0
merge_df.loc[merge_df['rating'] >= 4, 'label'] = 1

In [42]:
merge_df.head(3)

,rating,iid,parent_iid,uid,timestamp,reviews,main_category,title_review,average_rating,rating_number,description,categories,label
0,1.0,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,1.420266e+12,Definitely the weakest of the three albums SWV...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",0
1,1.0,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,1.379363e+12,Unlike It's About Time and New Beginning this ...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",0
2,4.0,B000002X4C,B000002X4C,AHAIZWWINONHYU7A3FVCEH75R4CA,9.534267e+11,"Their third and final album as a group, &quot;...",Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",1


In [43]:
merge_df.shape

(2130048, 13)

In [44]:
sample_df = merge_df.sample(n=100000, random_state=42)

In [45]:
user_counts = sample_df['uid'].value_counts()

In [46]:
user_counts.shape

(58188,)

In [ ]:
# encode user_id and asin
# merged_df['user_id_encoded'] = LabelEncoder().fit_transform(merged_df['user_id'])
# merged_df['asin_encoded'] = LabelEncoder().fit_transform(merged_df['asin'])
# merged_df['parent_asin_encoded'] = LabelEncoder().fit_transform(merged_df['parent_asin'])

In [47]:
min_user_interactions = 2
qualified_users = user_counts[user_counts >= min_user_interactions].index
filtered_df = sample_df[sample_df['uid'].isin(qualified_users)].copy()
print(f"After filtering users with >= {min_user_interactions} interactions: {len(filtered_df)} interactions")
print(f"Qualified users: {len(qualified_users)}")

After filtering users with >= 2 interactions: 60763 interactions
Qualified users: 18951


In [48]:
# sort to ensure we take the most recent interaction for each user
filtered_df = filtered_df.sort_values(['uid', 'timestamp'])
test_data = filtered_df.groupby('uid').tail(1).copy()
train_data = filtered_df[~filtered_df.index.isin(test_data.index)].copy()
print("\nLeave-One-Out split results:")
print(f"Train: {train_data.shape[0]} interactions, {train_data['uid'].nunique()} users")
print(f"Test: {test_data.shape[0]} interactions, {test_data['uid'].nunique()} users")


Leave-One-Out split results:
Train: 41812 interactions, 18951 users
Test: 18951 interactions, 18951 users


In [49]:
train_items = set(train_data['iid'].unique())
test_items = set(test_data['iid'].unique())

# calculate % overlap 
overlap = train_items.intersection(test_items)
print(f"Item overlap: {len(overlap)}/{len(test_items)} ({len(overlap)/len(test_items):.1%})")

Item overlap: 4821/16401 (29.4%)


In [50]:
print("\nAdditional Diagnostics:")
print(f"Average interactions per user (train): {train_data.shape[0] / train_data['uid'].nunique():.1f}")
print(f"Average interactions per user (test): {test_data.shape[0] / test_data['uid'].nunique():.1f}")

# save to CSV files
train_data.to_csv('train_set_new.csv', index=False)
test_data.to_csv('test_set_new.csv', index=False)


Additional Diagnostics:
Average interactions per user (train): 2.2
Average interactions per user (test): 1.0
